# Target Detection via Denoising Score Matching

Runs every experiment in the paper on the Pavia-University scene:

1. **IID single-class** background (Fig. 2)  — AMF, GMM-Levin, L-DART, **DART**, L-LRao, LRao
2. **IID multi-class** background (Fig. 3)
3. **Spatial / correlated** background (Table 1) — adds AMF-local, DART-CFAR, **DARTS**, DARTS-CFAR

Run top-to-bottom. Set `QUICK = True` for a fast smoke test, `False` for the full paper settings.


In [ ]:
# --- setup ---
import os, sys, glob, yaml
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Run from the repo root (the folder containing src/, configs/, data/).
# On Colab, clone the repo first and os.chdir into it.
ROOT = os.getcwd()
if os.path.basename(ROOT) == 'notebooks':
    ROOT = os.path.dirname(ROOT); os.chdir(ROOT)
sys.path.insert(0, ROOT)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
QUICK  = True          # True = fast smoke test; False = full paper settings
print('root =', ROOT, '| device =', DEVICE, '| QUICK =', QUICK)

def show_figs(run_dir, names):
    """Display the named PNG figures from <run_dir>/figures side by side."""
    fdir = os.path.join(run_dir, 'figures')
    for nm in names:
        p = os.path.join(fdir, nm)
        if not os.path.exists(p):
            print('(missing)', nm); continue
        plt.figure(figsize=(7, 4.2)); plt.imshow(mpimg.imread(p)); plt.axis('off'); plt.show()

## 1. IID — single-class background (Fig. 2)

In [ ]:
import src.iid as iid
cfg = yaml.safe_load(open('configs/iid_single.yaml'))
cfg['device'] = DEVICE
if QUICK:
    cfg.update(seed=42, n_train_list=[100, 500], rho_list=[0.01, 0.1],
               n_fixed_for_rho=200, dsm_epochs=50, lrao_epochs=20, test_size=400)
run_dir_single, _ = iid.run_iid(cfg, mode='single')
show_figs(run_dir_single, ['pauc_vs_n.png', 'pd_at_fa_vs_n.png', 'pdet_at_pfa_vs_rho.png'])

## 2. IID — multi-class background (Fig. 3)

In [ ]:
cfg = yaml.safe_load(open('configs/iid_multi.yaml'))
cfg['device'] = DEVICE
if QUICK:
    cfg.update(seed=42, n_train_list=[100, 500], rho_list=[0.01, 0.1],
               n_fixed_for_rho=200, dsm_epochs=50, lrao_epochs=20, test_size=400)
run_dir_multi, _ = iid.run_iid(cfg, mode='multi')
show_figs(run_dir_multi, ['pauc_vs_n.png', 'pd_at_fa_vs_n.png', 'pdet_at_pfa_vs_rho.png'])

## 3. Spatial — correlated background (Table 1)

Trains the global DART and the spatially adapted DARTS on disjoint train/test
boxes, then compares all detectors. `run_multiseed` averages over seeds and
prints the summary table; set `QUICK=True` for a single fast seed.

In [ ]:
import src.spatial as spatial
cfg = yaml.safe_load(open('configs/spatial.yaml'))
cfg['device'] = DEVICE
if QUICK:
    parent = spatial.run_from_cfg(cfg, dry_run=True)        # one fast seed
    spatial.show_plots_from_dir(parent, sub='foreign', inline=True)
else:
    parent = spatial.run_multiseed(cfg, seeds=(42, 43, 44, 45, 46))
    spatial.show_multiseed(parent, inline=True)
print('spatial results ->', parent)

### CFAR ablation (optional)

Sweep `cfar_lam` (local→global Fisher shrinkage) for the DART-CFAR / DARTS-CFAR
rows of Table 1. Re-uses the trained models via `--from-models` semantics in
`run_multiseed`; here we just re-run a couple of values for illustration.

In [ ]:
for lam in (0.1, 0.9):
    c = dict(cfg, cfar_lam=lam,
             active_detectors=['DART-CFAR', 'DARTS-CFAR', 'DARTS', 'AMF', 'GMM-Levin'])
    rd = spatial.run_from_cfg(c, dry_run=QUICK)
    print(f'cfar_lam={lam} ->', rd)